# Semana 5 · Clase 2 — Reúso de funciones, parámetro por defecto y segundo commit de Git

**Electiva de Concentración I (Ciencia de Datos) · 2026-II · Unidad 1**

El martes usted empaquetó en una sola función, `reconciliar_columna`, el mismo `for` que en la
semana 3 había copiado tres veces a mano; la guardó en el módulo `reconciliacion.py`, la **importó**
desde el cuaderno y dejó su **primer commit de Git**. El martes quedó una pregunta picando:

> *Si mañana llega una cuarta columna sucia —una que usted no había visto— ¿Qué tiene que hacer?*

Hoy llega esa cuarta columna. Descubrirá tres cosas que **profundizan** el trabajo del martes, sin herramienta nueva:

1. **El beneficio de haber escrito la función:** una columna nueva no se resuelve copiando otro `for`; sino **llamando** la función que ya tiene.
2. **Cómo se complementa una función sin romper lo que ya funciona** — un **parámetro con valor por defecto** — y cómo se **prueba** que no se dañó (comparar contra un resultado que ya conocía).
3. **Que Git no es un solo commit, es un hábito:** hoy hace su **segundo commit** y ve el historial crecer.

---
## Parte 0 — Recupere su función y sus columnas

Para no depender de tener abierto el cuaderno del martes, vuelva a pegar aquí —**letra por letra**—
la función `reconciliar_columna` **tal como la dejó el martes** (la versión de dos parámetros) y las
dos columnas de la semana 3. Corra la celda: todavía no hay nada que predecir, es el punto de partida
de hoy.

In [22]:
# La funcion del martes, tal cual (version de dos parametros).
def reconciliar_columna(columna_cruda, tipo_destino):
    """Recibe una columna sucia (lista de strings) y el tipo destino (int o float).
    Devuelve una lista nueva ya convertida: descarta 'n/d', repara la coma decimal
    y convierte cada valor con tipo_destino(valor)."""
    columna_limpia = []
    for valor in columna_cruda:
        if valor == 'n/d':
            continue
        valor = valor.replace(',', '.')
        columna_limpia.append(tipo_destino(valor))
    assert len(columna_limpia) <= len(columna_cruda), 'no puede sobrar mas datos limpios que crudos'
    assert len(columna_limpia) > 0, 'el resultado no puede quedar vacio'
    return columna_limpia

# Las mismas dos columnas de la semana 3.
bikers_crudo = ['16', '40', 'n/d', '1', '36', '34', '28', '9', '8', 'n/d', '57', '-5', '82', '92', '51', '34']
temp_crudo   = ['0,24', '0.22', '0.24', '0.24', '0.4', '0.4', '0,4', '0.24', '0.22', '0.22', '0.22', '0.2', '0.22', '0.2', '0.16', '0.4']

print('reconciliar_columna lista:', callable(reconciliar_columna))
print('bikers_crudo:', len(bikers_crudo), 'valores  |  temp_crudo:', len(temp_crudo), 'valores')

reconciliar_columna lista: True
bikers_crudo: 16 valores  |  temp_crudo: 16 valores


---
## Parte 1 — Llega una cuarta columna sucia

Una feria de emprendimiento registró, en una hoja de cálculo mal diligenciada, el **precio de
entrada** que cobró cada puesto (en pesos). La columna llegó como texto, con dos problemas: algunos
puestos no reportaron (`'n/d'`) y varios precios traen el símbolo `$` pegado al número.

Esta es exactamente la situación del martes: *una cuarta columna sucia*. Antes de escribir nada,
**prediga**: usted ya tiene una función que descarta `'n/d'` y convierte. ¿Basta con llamarla?

_Fallaría, lo que son los valores 'n/d' si los descartará pero los valores que tengan el simbolo '$' y como no son valores flotantes el codigo saltará error debibo a que nuestra función por el momento solo está procesando valores flotantes._

In [23]:
precios_crudo = ['$1200', '900', 'n/d', '$1500', '2000', 'n/d', '$800', '1100']
print(precios_crudo)

['$1200', '900', 'n/d', '$1500', '2000', 'n/d', '$800', '1100']


Prediga antes de correr qué pasa cuando le pasa `precios_crudo` a su función del martes con
`tipo_destino=int`: ¿devuelve `[1200, 900, 1500, ...]`, o falla? ¿Por qué?

In [24]:
reconciliar_columna(precios_crudo, int)

ValueError: invalid literal for int() with base 10: '$1200'

**`# TODO 1`** — La celda anterior **falló** (`ValueError`). No lo esconda: en una frase, explique

Cell In[7], line 1
----> 1 reconciliar_columna(precios_crudo, int)

Cell In[4], line 15, in reconciliar_columna(columna_cruda, tipo_destino, quitar)
     11             continue
     12         for caracter in quitar:          # si quitar == '', este for no hace nada
     13             valor = valor.replace(caracter, '')
     14         valor = valor.replace(',', '.')
---> 15         columna_limpia.append(tipo_destino(valor))
     16     assert len(columna_limpia) <= len(columna_cruda), 'no puede sobrar mas datos limpios que crudos'
     17     assert len(columna_limpia) > 0, 'el resultado no puede quedar vacio'
     18     return columna_limpia

ValueError: invalid literal for int() with base 10: '$1200'

**por qué** su función del martes no pudo con esta columna, conectándolo con lo que la función hace línea por línea (¿en qué paso exacto se rompe, y por qué `int('$1200')` no es válido pero `int('1200')`sí?). 

_Primero el valor $1200 no lo procesa como entero debido a que esta utilizando el simbolo de dinero, por lo cual la función no puede catalogarlo como entero._

---
## Parte 2 — Adaptar la función a los datos en la nueva columna sin dañar lo que ya servía

Uno podría pensar que sería fácil escribir *otra* función solo para precios. Pero entonces tendría dos funciones casi iguales — el mismo problema de copiar-pegar, disfrazado. Lo correcto es **extender** la función que ya tiene, de modo que:

- las llamadas viejas (`bikers`, `temp`) **sigan funcionando exactamente igual**, y
- la nueva necesidad (quitar el `$`) quede cubierta **solo cuando se pide**.

La herramienta para eso es un **parámetro con valor por defecto**: un parámetro nuevo, `quitar`, que vale `''` (nada) si no se dice lo contrario. Corra la celda; luego el `# TODO 2` le pide justificar por qué el valor por defecto es la clave.

In [25]:
def reconciliar_columna(columna_cruda, tipo_destino, quitar=''):
    """Recibe una columna sucia (lista de strings), el tipo destino (int o float) y,
    opcionalmente, una cadena 'quitar' con los caracteres a eliminar de cada valor
    antes de convertir (p. ej. '$'). Si 'quitar' no se pasa, vale '' y la funcion se
    comporta EXACTAMENTE como la version del martes.
    Devuelve una lista nueva: descarta 'n/d', quita los caracteres pedidos, repara la
    coma decimal y convierte con tipo_destino(valor)."""
    columna_limpia = []
    for valor in columna_cruda:
        if valor == 'n/d':
            continue
        for caracter in quitar:          # si quitar == '', este for no hace nada
            valor = valor.replace(caracter, '')
        valor = valor.replace(',', '.')
        columna_limpia.append(tipo_destino(valor))
    assert len(columna_limpia) <= len(columna_cruda), 'no puede sobrar mas datos limpios que crudos'
    assert len(columna_limpia) > 0, 'el resultado no puede quedar vacio'
    return columna_limpia

print('funcion redefinida (ahora con parametro quitar)')

funcion redefinida (ahora con parametro quitar)


In [26]:
# Ahora si: la MISMA funcion, pidiendole que quite el '$'.
precios = reconciliar_columna(precios_crudo, int, quitar='$')
print('precios reconciliados:', precios)
print('puestos con dato:', len(precios), '| precio total:', sum(precios), '| promedio:', round(sum(precios)/len(precios), 1))

precios reconciliados: [1200, 900, 1500, 2000, 800, 1100]
puestos con dato: 6 | precio total: 7500 | promedio: 1250.0


**La prueba que no se salta (honestidad).** Usted cambió la función. ¿Seguirá dando, para las
columnas viejas, **exactamente** lo mismo que ya sabía de la semana 3? No lo suponga: **pruébelo**.

In [27]:
# Las llamadas viejas NO pasan 'quitar' -> deben dar lo mismo que el martes.
bikers_limpio = reconciliar_columna(bikers_crudo, int)
temp_limpio   = reconciliar_columna(temp_crudo, float)

print('bikers -> n:', len(bikers_limpio), '| total:', sum(bikers_limpio),
      '| promedio:', round(sum(bikers_limpio)/len(bikers_limpio), 1))
print('  ¿coincide con la semana 3 (14, 483, 34.5)?',
      len(bikers_limpio) == 14 and sum(bikers_limpio) == 483)
print('temp   -> promedio:', round(sum(temp_limpio)/len(temp_limpio), 3),
      '| ¿coincide con la semana 3 (0.264)?', round(sum(temp_limpio)/len(temp_limpio), 3) == 0.264)

bikers -> n: 14 | total: 483 | promedio: 34.5
  ¿coincide con la semana 3 (14, 483, 34.5)? True
temp   -> promedio: 0.264 | ¿coincide con la semana 3 (0.264)? True


**`# TODO 2`** — Responda en dos frases:
1. ¿Por qué el valor por defecto `quitar=''` es lo que permite que `reconciliar_columna(bikers_crudo, int)`
   siga funcionando sin cambiarle nada, aunque la función ahora tenga tres parámetros?
2. Mire el `for caracter in quitar:` de adentro. Cuando `quitar` vale `''`, ¿Cuántas veces se ejecuta el ciclo y por qué eso deja el valor intacto?

_Frase 1: Porque lo que esta haciendo ese quitar se está especificando que quite el caracter de dinero, por lo tanto los valores que tengan ese caracter se le eliminara y se podra convertir más adelante a entero._
_Frase 2: El ciclo se ejecutará dependiendo del numero de datos que tenga esa lista, en este caso se podría ejecutar 8 veces, cuando no encuentra el caracter simplemente no pasará nada porque la variable convert no tendra nada que convertir._

**`# TODO 3` — Ahora usted.** Invente **su propia** cuarta columna sucia, de un dominio distinto al de
la feria (por ejemplo: número de libros prestados por día en una biblioteca, con algún `'n/d'`; o
temperaturas con coma decimal; o pesos con `$` **y** separador de miles, si se anima). Defínala en la
celda de abajo como una lista de strings y reconcíliela **con una sola llamada** a `reconciliar_columna`,
usando el `tipo_destino` y el `quitar` que correspondan. Imprima el resultado.

In [28]:
# TODO 3 -- su propia columna sucia + una sola llamada a reconciliar_columna
mi_columna_cruda = ["36KM", "40", "20", "26KM", "n/d", "n/d", "KM25", "45", "35,9KM", "20.8"]   # <- reemplace por su lista de strings
mi_columna = reconciliar_columna(mi_columna_cruda, float, quitar='KM')
print(mi_columna)


[36.0, 40.0, 20.0, 26.0, 25.0, 45.0, 35.9, 20.8]


---
## Parte 3 — Funciones que llaman funciones: un mini-resumen reutilizable

Cada vez que limpiamos una columna, terminamos calculando lo mismo a mano: cuántos datos quedaron, el total, el promedio, el mínimo, el máximo. Eso también se puede empaquetar — y lo interesante es que la función de resumen **no repite** la lógica de limpieza: **llama** a `reconciliar_columna`. Una función que usa otra función es la forma más simple de un *pipeline*: cada pieza hace una cosa.

In [29]:
def resumen_columna(columna_cruda, tipo_destino, quitar=''):
    """Reconcilia la columna (reusando reconciliar_columna) y devuelve un diccionario
    con las estadisticas basicas. No reimplementa la limpieza: la delega."""
    limpia = reconciliar_columna(columna_cruda, tipo_destino, quitar)
    return {
        'n': len(limpia),
        'total': sum(limpia),
        'promedio': round(sum(limpia) / len(limpia), 2),
        'minimo': min(limpia),
        'maximo': max(limpia),
    }

resumen_columna(bikers_crudo, int)

{'n': 14, 'total': 483, 'promedio': 34.5, 'minimo': -5, 'maximo': 92}

**`# TODO 4`** — En la celda de abajo:
1. Llame `resumen_columna` sobre `temp_crudo` (con `float`) y sobre `precios_crudo` (con `int` y
   `quitar='$'`), e imprima ambos diccionarios.
2. El resumen de `bikers` reporta `'minimo': -5`. Un conteo de bicicletas **no puede** ser negativo:
   ese `-5` es un dato sucio que sobrevivió a la reconciliación (la función quita `'n/d'` y símbolos,
   pero **no** juzga si un número tiene sentido). En una frase, diga a qué **etapa del proceso de datos**
   pertenece detectar ese `-5` y por qué *no* es tarea de `reconciliar_columna`.

temp_resumen = resumen_columna(temp_crudo, float)
print(temp_resumen)
precios_resumen = resumen_columna(precios_crudo, int, quitar='$')
print(precios_resumen)

_Creería que hace parte de la entrada de los datos, porque hacer que la función reconciliar_columna quite valores negativos listas como por ejemplo medición de temperatura estarían afectadas, por lo tanto es tarea de la captura de datos._

In [30]:
# TODO 4 -- resumen de temp_crudo y de precios_crudo
temp_resumen = resumen_columna(temp_crudo, float)
print(temp_resumen)
precios_resumen = resumen_columna(precios_crudo, int, quitar='$')
print(precios_resumen)

{'n': 16, 'total': 4.22, 'promedio': 0.26, 'minimo': 0.16, 'maximo': 0.4}
{'n': 6, 'total': 7500, 'promedio': 1250.0, 'minimo': 800, 'maximo': 2000}


---
## Parte 4 — numpy, un paso más: preguntarle al arreglo

El martes vio que `numpy` suma y promedia un arreglo completo de una sola vez (`.sum()`, `.mean()`),
sin `for`. Hoy un paso más: se le puede **preguntar** al arreglo *qué* valores cumplen una condición.
`bikers_np > promedio` no devuelve un solo `True/False`, sino un arreglo de `True/False` (una **máscara**). Usar esa máscara entre corchetes deja **solo** los valores que cumplen. Prediga cuántos días quedan por encima del promedio antes de correr.

In [31]:
import numpy as np

bikers_np = np.array(reconciliar_columna(bikers_crudo, int))
promedio = bikers_np.mean()

mascara = bikers_np > promedio          # arreglo de True/False, uno por dia
por_encima = bikers_np[mascara]         # deja solo los que cumplen

print('promedio:', promedio)
print('mascara :', mascara)
print('dias por encima del promedio:', por_encima)
print('cuantos:', por_encima.size, 'de', bikers_np.size)

promedio: 34.5
mascara : [False  True False  True False False False False  True False  True  True
  True False]
dias por encima del promedio: [40 36 57 82 92 51]
cuantos: 6 de 14


**`# TODO 5`** — En la celda de abajo, convierta `temp_crudo` en un arreglo de numpy y responda, con
una **máscara**, la pregunta inversa: ¿cuántos registros tienen temperatura **por debajo** del
promedio de la columna? Imprima el conteo y los valores. 

temp_np = np.array(reconciliar_columna(temp_crudo, float))
promedio = temp_np.mean()

mascara = temp_np < promedio
por_debajo = temp_np[mascara]

print(f'Promedio: {promedio}\nMascara: {mascara}\nTemperatura por debajo del promedio: {por_debajo}\nCuantos: {por_debajo.size} de {temp_np.size}')

In [35]:
# TODO 5 -- temp por debajo de su promedio, usando una mascara de numpy
temp_np = np.array(reconciliar_columna(temp_crudo, float))
promedio = temp_np.mean()

mascara = temp_np < promedio
por_debajo = temp_np[mascara]

print(f'Promedio: {promedio}\nMascara: {mascara}\nTemperatura por debajo del promedio: {por_debajo}\nCuantos: {por_debajo.size} de {temp_np.size}')

Promedio: 0.26375000000000004
Mascara: [ True  True  True  True False False False  True  True  True  True  True
  True  True  True False]
Temperatura por debajo del promedio: [0.24 0.22 0.24 0.24 0.24 0.22 0.22 0.22 0.2  0.22 0.2  0.16]
Cuantos: 12 de 16


---
### 🅢 Punto de encuentro 1 — de la función que creció al segundo commit *(pausa guiada)*

- El martes su módulo `reconciliacion.py` tenía **una** función de **dos** parámetros. Hoy la función
  **creció** (tercer parámetro `quitar`) y nació una **segunda** función (`resumen_columna`).
- Su archivo, entonces, **cambió**. Y un cambio sin registrar es un cambio que mañana no sabrá que hizo,
  ni podrá deshacer. Eso es justamente lo que Git resuelve — y por eso hoy hace su **segundo commit**.

*"El primer commit del martes fue una foto. Hoy va a tomar la segunda foto y ver que Git guarda el
álbum: `git log` mostrará **dos** commits, no uno."*

---
## Parte 5 — ⭐ El módulo crece y Git lo registra (segundo commit)

Todo esto pasa **fuera del cuaderno**, en su editor y en PowerShell. Es la parte que **no se recorta**:
Git se usa todos los días a partir de la semana 6.

**`# TODO 6` — Ahora usted (en el editor y en PowerShell):**

1. Abra `reconciliacion.py` (el que creó el martes, en la misma carpeta de sus cuadernos).
2. **Actualice** `reconciliar_columna` a la versión de hoy (la de tres parámetros, con `quitar`) y
   **agregue** debajo la función `resumen_columna` completa, tal como quedaron en las Partes 2 y 3.
3. Guarde el archivo.
4. En PowerShell, ubíquese (`cd`) en la carpeta del repositorio (donde el martes corrió `git init`) y:
   ```powershell
   git status                 # vera reconciliacion.py como "modified"
   git add reconciliacion.py
   git commit -m "amplia reconciliar_columna (parametro quitar) y agrega resumen_columna"
   git log --oneline          # ahora deben aparecer DOS commits
   ```

| Comando | Qué hace hoy |
|---|---|
| `git status` | Muestra que `reconciliacion.py` cambió desde el último commit (aparece como *modified*) |
| `git add reconciliacion.py` | Prepara **ese** archivo para el próximo commit |
| `git commit -m "..."` | Toma la **segunda foto**, con un mensaje que explica qué cambió |
| `git log --oneline` | Ahora lista **dos** commits: el del martes y el de hoy |

> **Nota honesta sobre el `import`.** Si el martes hizo `import reconciliacion` en un cuaderno y hoy
> cambió el `.py`, un `import` en el **mismo kernel** *no* recarga el archivo (Python lo recuerda). Por
> eso hoy trabajamos la función **dentro del cuaderno** y dejamos el `.py` para el commit. Cuando abra
> un cuaderno nuevo (o haga `Restart`), el `import` sí traerá la versión actualizada.

**`# TODO 7`** — Pegue aquí (en esta celda de texto) la salida real de su `git log --oneline`. Debe
mostrar **dos** commits: el primero del martes y el segundo de hoy. Si `git` le dio algún error en
cualquier paso, péguelo también y explique cómo lo resolvió — documentar el error puntúa.

```
(reemplace estas líneas con la salida de su terminal)
```

**Autochequeo:** corra `git show --stat HEAD` en PowerShell. ¿El commit de hoy lista `reconciliacion.py`
como archivo cambiado? Si no aparece, probablemente olvidó `git add reconciliacion.py` antes del commit
— repítalo (un tercer commit está perfectamente bien).

---
## Cierre

Antes de subir a Brightspace, verifique:

1. **`Kernel → Restart & Run All`** corre de principio a fin sin errores (excepto el `ValueError` de la
   Parte 1, que **debe** aparecer: es parte del ejercicio) y guarde el cuaderno **con las salidas visibles**.
2. Los cinco `# TODO` están respondidos: el porqué del fallo (1), el valor por defecto (2), su propia
   columna sucia reconciliada (3), los resúmenes y el caso del `-5` (4) y la máscara de numpy (5).
3. `reconciliacion.py` quedó actualizado y su `git log --oneline` muestra **dos** commits.
4. Suba **este cuaderno** (`.ipynb`, ya ejecutado) a Brightspace **antes del cierre de la semana**.

> **Cliffhanger para la semana 6.** Hoy cargó cada columna a mano, letra por letra, y reconcilió una por
> una. Pero los datos reales no llegan como cuatro listas escritas a mano: llegan como un **archivo**
> con muchas columnas a la vez. La semana entrante conocerá la herramienta que carga ese archivo entero
> de un solo `read_csv` y lo trata como una tabla — **pandas**. Su `reconciliar_columna` no se pierde:
> es exactamente la intuición de lo que pandas hará por usted, columna por columna.
